<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: The Silent Failure

**Step 1 (run unchanged):** With `amount_requested = 0`, the original prints **nothing**. That is not an error, it is a **bug**: Python ran to the end, exited with status 0, and the second outcome had no answer, so silence looks exactly like "this message is fine".

**Step 2:** Add the `else` and the count comment.

In [ ]:
# CODE_RUNNER: 3.6 Popcorn 1 - Run it first. Then make both outcomes print.

amount_requested = 0

# 1 condition -> 2 outcomes -> 2 answers
if amount_requested > 100:
    print("This message is asking for a large amount of money.")
else:
    print("This message is not asking for a large amount of money.")

**Actual output:** `This message is not asking for a large amount of money.` Change `amount_requested` to `500` and the first message prints instead, so both outcomes now have an answer.

## Popcorn Hack 2: The Chain in the Wrong Order

**Step 1 (run unchanged):** With `risk_score = 9` it prints `Be a little careful, but this is probably fine.` The trainer tells the person with the *worst* possible score that everything is fine.

**Step 2, unreachable branches:** the `elif risk_score >= 6` branch and the `elif risk_score >= 9` branch can **never** run, for any score. Any score that is 6 or 9 is already `>= 3`, so the first `if` catches it. `elif` only gets asked when everything above it was `False`.

**Step 3 fix:** order thresholds narrowest first, highest threshold first.

In [ ]:
# CODE_RUNNER: 3.6 Popcorn 2 - Run it first. Then fix the order.

risk_score = 9

# 3 thresholds -> 4 outcomes -> 4 answers
if risk_score >= 9:
    print("Do not reply. This is almost certainly a scam.")
elif risk_score >= 6:
    print("This looks risky. Check with someone you trust before replying.")
elif risk_score >= 3:
    print("Be a little careful, but this is probably fine.")
else:
    print("Nothing suspicious in this message.")

**Actual output:** `Do not reply. This is almost certainly a scam.`

Counting answers would not have caught the original bug (it had 4 answers for 4 outcomes). What catches it is asking whether each branch is *reachable*.

## Homework Hack: Score the Message, Then Advise

**Scoring step:** a separate `if` for each signal (+4 gift cards, +3 amount over 100, +2 stranger). **Reporting step:** an `if`/`elif`/`else` chain with four answers, ordered highest threshold first.

In [ ]:
# CODE_RUNNER: 3.6 Homework - score a message, then advise on it

message = "URGENT: send gift cards now to unlock your account"
amount_requested = 500
known_contact = False

risk_score = 0

# Each signal is its own separate question, and each one only ADDS to the score.
# No else is needed: when the signal is absent, there is nothing to add, and the
# score is reported in the chain below, so nothing is left silent.
if "gift card" in message:
    risk_score = risk_score + 4

if amount_requested > 100:
    risk_score = risk_score + 3

if not known_contact:
    risk_score = risk_score + 2

print("Risk score:", risk_score)

# 3 thresholds -> 4 outcomes -> 4 answers
if risk_score >= 9:
    print("Do not reply. This is almost certainly a scam.")
elif risk_score >= 6:
    print("This looks risky. Check with someone you trust before replying.")
elif risk_score >= 3:
    print("Be a little careful, but this is probably fine.")
else:
    print("Nothing suspicious in this message.")

**Actual output:** `Risk score: 9`, then `Do not reply. This is almost certainly a scam.`

## Reachability Proof

The maximum score is 4 + 3 + 2 = 9, and the possible totals are 0, 2, 3, 4, 5, 6, 7, and 9. One set of inputs for each answer:

| Answer | Inputs | Score |
|--------|--------|-------|
| `Do not reply...` (>= 9) | message with "gift card", amount 500, unknown sender | 9 |
| `This looks risky...` (>= 6) | message with "gift card", amount 500, **known** sender | 7 |
| `Be a little careful...` (>= 3) | message with "gift card", amount 0, **known** sender | 4 |
| `Nothing suspicious...` (else) | no "gift card", amount 0, **known** sender | 0 |

Every branch has inputs that reach it, so no branch is dead code.

## Tests: The Extremes and Every Branch

The same scoring and chain, run on four messages: completely clean, one that trips every signal, and the two in-between cases from the reachability table. Every outcome prints something.

In [ ]:
# CODE_RUNNER: 3.6 Homework Tests - clean, in-between, and all-signals messages

tests = [
    ("Hi Mom, are we still on for Sunday?", 0, True),
    ("Please send a gift card code", 0, True),
    ("Please send a gift card code", 500, True),
    ("URGENT: send gift cards now to unlock your account", 500, False),
]

for message, amount_requested, known_contact in tests:
    risk_score = 0

    if "gift card" in message:
        risk_score = risk_score + 4

    if amount_requested > 100:
        risk_score = risk_score + 3

    if not known_contact:
        risk_score = risk_score + 2

    print("Score", risk_score, "->", end=" ")

    # 3 thresholds -> 4 outcomes -> 4 answers
    if risk_score >= 9:
        print("Do not reply. This is almost certainly a scam.")
    elif risk_score >= 6:
        print("This looks risky. Check with someone you trust before replying.")
    elif risk_score >= 3:
        print("Be a little careful, but this is probably fine.")
    else:
        print("Nothing suspicious in this message.")

**Expected output:**

```
Score 0 -> Nothing suspicious in this message.
Score 4 -> Be a little careful, but this is probably fine.
Score 7 -> This looks risky. Check with someone you trust before replying.
Score 9 -> Do not reply. This is almost certainly a scam.
```

**Stretch:** the scoring `if`s have no `else` on purpose because each one only *adds* to a running total, and the score is always reported afterward by a chain that ends in `else`. In Example A the missing `else` left the *final answer* silent, which is why it was a bug there and is fine here.

**Submission notes line:** `3.6 Conditionals popcorn + homework complete`